# V10 Training Monitor

Use this notebook while `train_v10.py` is running. It reads the live telemetry files written into the run directory each epoch:

- `epoch_history.csv` for charts
- `latest_status.json` for the current state
- `training_cohort.json` for the curated shard budget
- `previews/*.png` for the latest qualitative output

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from PIL import Image

RUN_DIR = Path(r"i:\parp\parp-tools\output\ml-training\v10_large_500ep")
CSV_PATH = RUN_DIR / "epoch_history.csv"
STATUS_PATH = RUN_DIR / "latest_status.json"
COHORT_PATH = RUN_DIR / "training_cohort.json"
PREVIEW_DIR = RUN_DIR / "previews"

if not RUN_DIR.exists():
    raise FileNotFoundError(f"Run directory not found: {RUN_DIR}")

plt.style.use("seaborn-v0_8-darkgrid")
RUN_DIR

In [ ]:
history = pd.read_csv(CSV_PATH) if CSV_PATH.exists() else pd.DataFrame()
status = json.loads(STATUS_PATH.read_text(encoding="utf-8")) if STATUS_PATH.exists() else {}
cohort = json.loads(COHORT_PATH.read_text(encoding="utf-8")) if COHORT_PATH.exists() else {}

display(Markdown(f"## Run: `{RUN_DIR}`"))
display(Markdown(
    f"Best no-WDL val: **{status.get('best_val_loss', float('nan')):.6f}** at epoch **{status.get('best_epoch', 0)}**  \
Latest epoch: **{status.get('latest_epoch', 0)}**  \
Current stall: **{status.get('stall', 0)}**  \
Selected/train/val samples: **{status.get('selected_samples', 0)} / {status.get('train_samples', 0)} / {status.get('val_samples', 0)}**"
))

if not history.empty:
    display(history.tail(10))
else:
    display(Markdown("No `epoch_history.csv` found yet. Start or resume the trainer first."))

if cohort:
    by_build = pd.DataFrame(sorted(cohort.get('by_build', {}).items()), columns=['build', 'samples'])
    by_map = pd.DataFrame(sorted(cohort.get('by_map', {}).items()), columns=['map', 'samples'])
    display(Markdown("### Cohort mix"))
    display(by_build.sort_values('samples', ascending=False).head(12))
    display(by_map.sort_values('samples', ascending=False).head(12))

In [ ]:
if history.empty:
    raise RuntimeError("No epoch history available yet.")

fig, axes = plt.subplots(3, 1, figsize=(14, 14), sharex=True)

axes[0].plot(history['epoch'], history['train_loss'], label='train_loss', linewidth=2)
axes[0].plot(history['epoch'], history['val_no_wdl_loss'], label='val_no_wdl_loss', linewidth=2)
if 'val_real_wdl_loss' in history:
    axes[0].plot(history['epoch'], history['val_real_wdl_loss'], label='val_real_wdl_loss', linewidth=1.5)
axes[0].scatter(history.loc[history['is_best'] == True, 'epoch'], history.loc[history['is_best'] == True, 'val_no_wdl_loss'], color='tab:red', label='best epochs', zorder=3)
axes[0].set_ylabel('loss')
axes[0].set_title('Loss curves')
axes[0].legend()

axes[1].plot(history['epoch'], history['learning_rate'], color='tab:green', linewidth=2)
axes[1].set_yscale('log')
axes[1].set_ylabel('learning rate')
axes[1].set_title('Scheduler trace')

axes[2].plot(history['epoch'], history['epoch_seconds'], label='epoch_seconds', linewidth=2)
if 'avg_epoch_seconds' in history:
    axes[2].plot(history['epoch'], history['avg_epoch_seconds'], label='avg_epoch_seconds', linewidth=2)
if 'eta_seconds' in history:
    axes[2].plot(history['epoch'], history['eta_seconds'], label='eta_seconds', linewidth=1.5)
if 'train_samples_per_second' in history:
    ax2b = axes[2].twinx()
    ax2b.plot(history['epoch'], history['train_samples_per_second'], color='tab:orange', alpha=0.7, label='train_samples_per_second')
    ax2b.set_ylabel('samples/sec')
axes[2].set_ylabel('seconds')
axes[2].set_xlabel('epoch')
axes[2].set_title('Runtime and throughput')
axes[2].legend(loc='upper left')

fig.tight_layout()
plt.show()

display(history[['epoch', 'val_no_wdl_loss', 'best_val_loss', 'stall', 'learning_rate', 'epoch_seconds', 'train_samples_per_second']].tail(15))

In [ ]:
preview_paths = sorted(PREVIEW_DIR.glob('*.png'))
if not preview_paths:
    raise RuntimeError(f"No preview PNGs found in {PREVIEW_DIR}")

columns = 2
rows = (len(preview_paths) + columns - 1) // columns
fig, axes = plt.subplots(rows, columns, figsize=(14, 5 * rows))
axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

for axis, preview_path in zip(axes, preview_paths):
    axis.imshow(Image.open(preview_path))
    axis.set_title(preview_path.name)
    axis.axis('off')

for axis in axes[len(preview_paths):]:
    axis.axis('off')

fig.suptitle('Latest validation previews', fontsize=16)
fig.tight_layout()
plt.show()